In [1]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

In [2]:
import warnings
import pandas as pd
import numpy as np
import pyodbc
import random
from scipy.spatial import distance
from numpy.linalg import inv
from sklearn import preprocessing
from scipy.spatial.distance import squareform
from scipy.spatial.distance import pdist
import datetime
import matplotlib.pyplot as plt

In [3]:
con = pyodbc.connect(r'Driver={Microsoft Access Driver (*.mdb, *.accdb)};'
                     r'DBQ=C:\Users\...\Python Files\...\Tyson Sales\Tyson Frozen Foods - Final.accdb')
sql = "SELECT * FROM INVOICE_FACT WHERE INVOICE_DAY_KEY BETWEEN 827045 AND 827047"
original_data = pd.read_sql(sql,con)
data = original_data.copy()
data.sort_values(by='PRODUCT_KEY').head(100)

,INVOICE_DAY_KEY,SALES_ORDER_NUMBER,SALES_ORDER_LINE_NUMBER,CUSTOMER_KEY,SHIP_DAY_KEY,PRODUCT_KEY,SHIP_LOCATION_KEY,INVOICE_NUMBER,CASES_ORDERED,POUNDS_ORDERED,CASES_SHIPPED,POUNDS_SHIPPED,SALES_AMOUNT,DISCOUNT1_AMOUNT,DISCOUNT2_AMOUNT,SHIP_COST_AMOUNT
9561,827045.0,4201864.0,1.0,25183.0,827659.0,3.0,68.0,5909503.0,1.0,30.0,1.0,30.0,29.45,0.00,0.0,3.78
6440,827045.0,2859130.0,51.0,51517.0,827104.0,3.0,68.0,4660490.0,11.0,330.0,11.0,330.0,640.20,0.00,0.0,35.39
15636,827047.0,9173601.0,1.0,18516.0,825780.0,7.0,191.0,1424172.0,1.0,47440.0,1.0,47440.0,6717.50,-161.30,0.0,0.00
15946,827047.0,9310007.0,1.0,18516.0,825822.0,7.0,191.0,1538976.0,1.0,47180.0,1.0,47180.0,6369.30,-160.41,0.0,0.00
13145,827047.0,7940646.0,1.0,18476.0,825372.0,12.0,68.0,348533.0,1.0,48460.0,1.0,48460.0,4492.24,-828.67,0.0,855.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,827045.0,368718.0,10.0,39991.0,826142.0,127.0,68.0,2388255.0,65.0,780.0,65.0,780.0,2160.60,0.00,0.0,42.47
15288,827045.0,9014055.0,2.0,25401.0,825729.0,127.0,140.0,1289333.0,26.0,312.0,26.0,312.0,817.44,0.00,0.0,10.98
1098,827045.0,575947.0,12.0,46581.0,826213.0,127.0,68.0,2573072.0,65.0,780.0,65.0,780.0,2316.60,0.00,0.0,34.10
11459,827047.0,7046088.0,2.0,54181.0,1318.0,127.0,140.0,412988.0,39.0,468.0,39.0,468.0,1024.92,-2.57,0.0,13.72


In [4]:
Sales_Error = 70000
Discount1_Error = 10000
Discount2_Error = 10000
Shipping_Error = 1000

Rounds = 100
Size = 100
Similarity_Threshold = 0
Simulation = 1

_Similarity_Threshold = []
_Simulation = []
_Investigated_Exceptions_Count = []
_Similar_Exceptions_Count = []
_Lowest_Risk_Selected = []

_Positive_Inference_Count = []
_False_Positive_Inference_Count = []
_Negative_Inference_Count = []
_False_Negative_Inference_Count = []

_Sales_Misstatement_Amount = []
_Discount1_Misstatement_Amount = []
_Discount2_Misstatement_Amount = []
_Shipping_Misstatement_Amount = []
_Account_Misstated_Amount = []

_Sales_Investigation_Restatement_Amount = []
_Discount1_Investigation_Restatement_Amount = []
_Discount2_Investigation_Restatement_Amount = []
_Shipping_Investigation_Restatement_Amount = []
_Investigation_Restatement_Amount = []

_Sales_Inference_Restatement_Amount = []
_Discount1_Inference_Restatement_Amount = []
_Discount2_Inference_Restatement_Amount = []
_Shipping_Inference_Restatement_Amount = []
_Inference_Restatement_Amount = []

_Remaining_Sales_Misstatement_Amount = []
_Remaining_Discount1_Misstatement_Amount = []
_Remaining_Discount2_Misstatement_Amount = []
_Remaining_Shipping_Misstatement_Amount = []
_Remaining_Misstated_Amount = []

In [5]:
warnings.filterwarnings("ignore")
while Similarity_Threshold <= 0.02:
    while Simulation <= Rounds:
        random.seed(Simulation)
        pivot_count = pd.pivot_table(data[data['SALES_AMOUNT'] > 0], index='PRODUCT_KEY', values='SHIP_LOCATION_KEY', aggfunc='count')
        usual_products = pivot_count[pivot_count['SHIP_LOCATION_KEY'] > 10]
        Error_Candidate_Products = list(usual_products.index)
        Sales_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)
        random.shuffle(Error_Candidate_Products)
        Discount1_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)
        random.shuffle(Error_Candidate_Products)
        Discount2_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)
        random.shuffle(Error_Candidate_Products)
        Shipping_Error_Products = random.Random(Simulation).sample(Error_Candidate_Products, 5)

        Sales_Errors = []
        for product in Sales_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Sales_Errors.extend(errors)

        Discount1_Errors = []
        for product in Discount1_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Discount1_Errors.extend(errors)

        Discount2_Errors = []
        for product in Discount2_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Discount2_Errors.extend(errors)

        Shipping_Errors = []
        for product in Shipping_Error_Products:
            errors = random.Random(Simulation).sample(list(data[(data['SALES_AMOUNT'] > 0) & (data['PRODUCT_KEY'] == product)].index), 5)
            Shipping_Errors.extend(errors)    

        All_Errors = set(set(Sales_Errors) | set(Discount1_Errors) | set(Discount2_Errors) | set(Shipping_Errors)) 
        Seeded_Errors_Data = data.copy()
        Seeded_Errors_Data['Original_SALES_AMOUNT'] = Seeded_Errors_Data['SALES_AMOUNT']
        Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['DISCOUNT1_AMOUNT']
        Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['DISCOUNT2_AMOUNT']
        Seeded_Errors_Data['Original_SHIP_COST_AMOUNT'] = Seeded_Errors_Data['SHIP_COST_AMOUNT']

        count = 0
        while count <= 4:
            Seeded_Errors_Data.loc[Sales_Errors[count*5:(count+1)*5], 'SALES_AMOUNT'] = Seeded_Errors_Data['SALES_AMOUNT'] + Sales_Error*(1+count*0.20)
            Seeded_Errors_Data.loc[Discount1_Errors[count*5:(count+1)*5], 'DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['DISCOUNT1_AMOUNT'] + Discount1_Error*(1+count*0.25)
            Seeded_Errors_Data.loc[Discount2_Errors[count*5:(count+1)*5], 'DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['DISCOUNT2_AMOUNT'] + Discount2_Error*(1+count*0.25)
            Seeded_Errors_Data.loc[Shipping_Errors[count*5:(count+1)*5], 'SHIP_COST_AMOUNT'] = Seeded_Errors_Data['SHIP_COST_AMOUNT'] + Shipping_Error*(1+count*0.10)
            count += 1

        Original_net_sales = sum(Seeded_Errors_Data[['Original_SALES_AMOUNT', 'Original_DISCOUNT1_AMOUNT', 'Original_DISCOUNT2_AMOUNT', 'Original_SHIP_COST_AMOUNT']].sum(axis=1))
        Misstated_net_sales = sum(Seeded_Errors_Data[['SALES_AMOUNT', 'DISCOUNT1_AMOUNT', 'DISCOUNT2_AMOUNT', 'SHIP_COST_AMOUNT']].sum(axis=1))

        #Filter 1: Improper Sales Amounts
        Seeded_Errors_Data['Sales_Filter'] = np.where(Seeded_Errors_Data['SALES_AMOUNT'] <= 10000, 0, Seeded_Errors_Data['SALES_AMOUNT']/1000)
        #Filter 2: Improper Discount 1 Amounts
        Seeded_Errors_Data['Discount1_Filter'] = np.where(Seeded_Errors_Data['DISCOUNT1_AMOUNT'] <= 1000, 0, Seeded_Errors_Data['DISCOUNT1_AMOUNT']/100)
        #Filter 3: Improper Discount 2 Amounts
        Seeded_Errors_Data['Discount2_Filter'] = np.where(Seeded_Errors_Data['DISCOUNT2_AMOUNT'] <= 1000, 0, Seeded_Errors_Data['DISCOUNT2_AMOUNT']/100)
        #Filter 4: Improper Shipping Cost Amounts
        Seeded_Errors_Data['Shipping_Filter'] = np.where(Seeded_Errors_Data['SALES_AMOUNT'] <= 0, 0, np.where(Seeded_Errors_Data['SHIP_COST_AMOUNT']/Seeded_Errors_Data['SALES_AMOUNT'] >= 0.5,
                                                                                                              Seeded_Errors_Data['SHIP_COST_AMOUNT']/Seeded_Errors_Data['SALES_AMOUNT']/0.5,
                                                                                                              np.where(Seeded_Errors_Data['SHIP_COST_AMOUNT'] >=1000, 
                                                                                                                       Seeded_Errors_Data['SHIP_COST_AMOUNT']/100, 0)))

        CovMat = Seeded_Errors_Data[['Sales_Filter', 'Discount1_Filter', 'Discount2_Filter', 'Shipping_Filter']].cov()
        Seeded_Errors_Data['Risk_Score'] = distance.mahalanobis((pd.Series([Seeded_Errors_Data['Sales_Filter'], Seeded_Errors_Data['Discount1_Filter'], 
                                                                            Seeded_Errors_Data['Discount2_Filter'], Seeded_Errors_Data['Shipping_Filter']])),
                                                                [0, 0, 0, 0], CovMat)

        Seeded_Errors_Data = Seeded_Errors_Data.sort_values(by='Risk_Score', ascending=False)

        min_max_scaler = preprocessing.MinMaxScaler()
        #Seeded_Errors_Data['Risk_Score'] = min_max_scaler.fit_transform(np.array(Seeded_Errors_Data['Risk_Score']).reshape(-1,1))
        
        Exception_Data = Seeded_Errors_Data[Seeded_Errors_Data['Risk_Score'] >= 0]
        Similarity_Data = Seeded_Errors_Data[['CASES_ORDERED', 'POUNDS_ORDERED', 'CASES_SHIPPED', 'POUNDS_SHIPPED', 'SALES_AMOUNT', 'DISCOUNT1_AMOUNT', 'DISCOUNT2_AMOUNT', 'SHIP_COST_AMOUNT',  
                                              'Sales_Filter', 'Discount1_Filter', 'Discount2_Filter', 'Shipping_Filter']]
        x = Similarity_Data.values #returns a numpy array
        x_scaled = min_max_scaler.fit_transform(x)
        Similarity_Data_Normalized = pd.DataFrame(x_scaled)
        TranID = list(Similarity_Data.index.tolist())
        Data_Columns = list(Similarity_Data.columns.tolist())
        Similarity_Data_Normalized.columns = Data_Columns
        Similarity_Data_Normalized = Similarity_Data_Normalized.set_index([TranID])
        Similarity_Data_Normalized_Matrix = pd.DataFrame(squareform(pdist(Similarity_Data_Normalized)))
        Similarity_Data_Normalized_Matrix = Similarity_Data_Normalized_Matrix.set_index([TranID])
        Similarity_Data_Normalized_Matrix.columns = TranID
        np.fill_diagonal(Similarity_Data_Normalized_Matrix.values, np.nan)

        Exception_List = Seeded_Errors_Data[['Risk_Score']].sort_values(by='Risk_Score', ascending=False)
        Skipper_Selected_Exceptions_Scores = Exception_List.iloc[0:0]
        Skipped_Exceptions = []
        select = Exception_List.loc[Exception_List['Risk_Score'].idxmax()]
        Skipper_Selected_Exceptions_Scores = Skipper_Selected_Exceptions_Scores.append(select)
        Exception_List = Exception_List.drop(select.to_frame())

        MinEucDis = Similarity_Threshold
        n = 1
        while len(Exception_List) > 0 and n < Size:
            select = Exception_List.loc[Exception_List['Risk_Score'].idxmax()]
            temp_cs_data = Similarity_Data_Normalized_Matrix.loc[[select.name]]
            temp_cs_data = temp_cs_data[Skipper_Selected_Exceptions_Scores.index]
            if temp_cs_data.to_numpy().min() >= MinEucDis: 
                Skipper_Selected_Exceptions_Scores = Skipper_Selected_Exceptions_Scores.append(select)
                Exception_List = Exception_List.drop(select.to_frame())
                n += 1
            else:
                Skipped_Exceptions.append(temp_cs_data.index[0])
                Exception_List = Exception_List.drop(select.to_frame())

        Skipper_Selected_Exceptions = list(Skipper_Selected_Exceptions_Scores.index)
        Skipper_Similar_Exceptions = []
        for exception in Skipper_Selected_Exceptions:
            select_similarity = Similarity_Data_Normalized_Matrix[Similarity_Data_Normalized_Matrix[exception] < MinEucDis]
            Skipper_Similar_Exceptions = list(Skipper_Similar_Exceptions + list(select_similarity.index))
        Skipper_Similar_Exceptions = set(Skipper_Similar_Exceptions) - set(Skipper_Selected_Exceptions)
        Skipper_Similar_Exceptions = list(Skipper_Similar_Exceptions)

        Restated_for_Sales = list(set(Skipper_Selected_Exceptions) & set(Sales_Errors))
        Restated_for_Discount1 = list(set(Skipper_Selected_Exceptions) & set(Discount1_Errors))
        Restated_for_Discount2 = list(set(Skipper_Selected_Exceptions) & set(Discount2_Errors))
        Restated_for_Shipping = list(set(Skipper_Selected_Exceptions) & set(Shipping_Errors))
        
        Seeded_Errors_Data['Restated_SALES_AMOUNT'] = Seeded_Errors_Data['SALES_AMOUNT']
        Seeded_Errors_Data['Restated_DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['DISCOUNT1_AMOUNT']
        Seeded_Errors_Data['Restated_DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['DISCOUNT2_AMOUNT']
        Seeded_Errors_Data['Restated_SHIP_COST_AMOUNT'] = Seeded_Errors_Data['SHIP_COST_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Sales, 'Restated_SALES_AMOUNT'] = Seeded_Errors_Data['Original_SALES_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Discount1, 'Restated_DISCOUNT1_AMOUNT'] = Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Discount2, 'Restated_DISCOUNT2_AMOUNT'] = Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT']
        Seeded_Errors_Data.loc[Restated_for_Shipping, 'Restated_SHIP_COST_AMOUNT'] = Seeded_Errors_Data['Original_SHIP_COST_AMOUNT']
        
        Similarity_Error_Rate = pd.DataFrame()
        Similar_Exception = []
        Closest_Selected_Exception = []
        Similar_Exception_Error_Pattern = []
        Closest_Selected_Exception_Error_Pattern = []

        for similar in Skipper_Similar_Exceptions:
            Closest_Selected = Similarity_Data_Normalized_Matrix.loc[[similar]][Skipper_Selected_Exceptions].idxmin(axis=1).to_numpy()[0]
            Similar_Error_Pattern = set()
            Closest_Error_Pattern = set()

            if similar in Sales_Errors:
                Similar_Error_Pattern.add(1)
            if similar in Discount1_Errors:
                Similar_Error_Pattern.add(2)
            if similar in Discount2_Errors:
                Similar_Error_Pattern.add(3)
            if similar in Shipping_Errors:
                Similar_Error_Pattern.add(4)

            if Closest_Selected in Sales_Errors:
                Closest_Error_Pattern.add(1)
            if Closest_Selected in Discount1_Errors:
                Closest_Error_Pattern.add(2)
            if Closest_Selected in Discount2_Errors:
                Closest_Error_Pattern.add(3)
            if Closest_Selected in Shipping_Errors:
                Closest_Error_Pattern.add(4)

            Similar_Exception.append(similar)
            Closest_Selected_Exception.append(Closest_Selected)
            Similar_Exception_Error_Pattern.append(Similar_Error_Pattern)
            Closest_Selected_Exception_Error_Pattern.append(Closest_Error_Pattern)

        Similarity_Error_Rate['Similar Exception'] = Similar_Exception
        Similarity_Error_Rate['Closest Selected Exception'] = Closest_Selected_Exception
        Similarity_Error_Rate['Similar Exception Error Pattern'] = Similar_Exception_Error_Pattern
        Similarity_Error_Rate['Closest Selected Exception Error Pattern'] = Closest_Selected_Exception_Error_Pattern

        Sales_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Sales_Error_Amount = Seeded_Errors_Data.loc[closest][['SALES_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_SALES_AMOUNT']][0]
            Sales_Error_Amount_List.append(Sales_Error_Amount)
        Discount1_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Discount1_Error_Amount = Seeded_Errors_Data.loc[closest][['DISCOUNT1_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_DISCOUNT1_AMOUNT']][0]
            Discount1_Error_Amount_List.append(Discount1_Error_Amount)
        Discount2_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Discount2_Error_Amount = Seeded_Errors_Data.loc[closest][['DISCOUNT2_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_DISCOUNT2_AMOUNT']][0]
            Discount2_Error_Amount_List.append(Discount2_Error_Amount)
        Shipping_Error_Amount_List = []
        for closest in Similarity_Error_Rate['Closest Selected Exception']:
            Shipping_Error_Amount = Seeded_Errors_Data.loc[closest][['SHIP_COST_AMOUNT']][0] - Seeded_Errors_Data.loc[closest][[ 'Original_SHIP_COST_AMOUNT']][0]
            Shipping_Error_Amount_List.append(Shipping_Error_Amount)

        Similarity_Error_Rate['Sales_Error'] = Sales_Error_Amount_List
        Similarity_Error_Rate['Discount1_Error'] = Discount1_Error_Amount_List
        Similarity_Error_Rate['Discount2_Error'] = Discount2_Error_Amount_List
        Similarity_Error_Rate['Shipping_Error'] = Shipping_Error_Amount_List

        Adjustment_Table = Similarity_Error_Rate[['Similar Exception', 'Sales_Error', 'Discount1_Error', 'Discount2_Error', 'Shipping_Error']]
        Adjustment_Table = Adjustment_Table.set_index('Similar Exception')
        Adjustment_Table = Adjustment_Table.rename(columns={"Sales_Error": "Restated_SALES_AMOUNT", "Discount1_Error": "Restated_DISCOUNT1_AMOUNT", "Discount2_Error": "Restated_DISCOUNT2_AMOUNT", 
                                                            "Shipping_Error": "Restated_SHIP_COST_AMOUNT"})
        Adjustment_Table = -Adjustment_Table
        Seeded_Errors_Data = Seeded_Errors_Data.add(Adjustment_Table, fill_value=0)

        _Similarity_Threshold.append(Similarity_Threshold)
        _Simulation.append(Simulation)
        _Investigated_Exceptions_Count.append(len(Skipper_Selected_Exceptions))
        _Similar_Exceptions_Count.append(len(Similarity_Error_Rate))
        _Lowest_Risk_Selected.append(min(Skipper_Selected_Exceptions_Scores['Risk_Score']))
        _Positive_Inference_Count.append(len(Similarity_Error_Rate[(Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                   (Similarity_Error_Rate['Similar Exception Error Pattern'] == Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))
        _False_Positive_Inference_Count.append(len(Similarity_Error_Rate[(Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                         (Similarity_Error_Rate['Similar Exception Error Pattern'] != Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))
        _Negative_Inference_Count.append(len(Similarity_Error_Rate[(~Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                   (Similarity_Error_Rate['Similar Exception Error Pattern'] == Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))
        _False_Negative_Inference_Count.append(len(Similarity_Error_Rate[(~Similarity_Error_Rate['Similar Exception Error Pattern'].astype(bool)) & 
                                                                         (Similarity_Error_Rate['Similar Exception Error Pattern'] != Similarity_Error_Rate['Closest Selected Exception Error Pattern'])]))

        _Sales_Misstatement_Amount.append(sum(Seeded_Errors_Data['SALES_AMOUNT'] - Seeded_Errors_Data['Original_SALES_AMOUNT']))
        _Discount1_Misstatement_Amount.append(sum(Seeded_Errors_Data['DISCOUNT1_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT']))
        _Discount2_Misstatement_Amount.append(sum(Seeded_Errors_Data['DISCOUNT2_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT']))
        _Shipping_Misstatement_Amount.append(sum(Seeded_Errors_Data['SHIP_COST_AMOUNT'] - Seeded_Errors_Data['Original_SHIP_COST_AMOUNT']))

        _Sales_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'SALES_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'Restated_SALES_AMOUNT']))
        _Discount1_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'DISCOUNT1_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'Restated_DISCOUNT1_AMOUNT']))
        _Discount2_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'DISCOUNT2_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'Restated_DISCOUNT2_AMOUNT']))
        _Shipping_Investigation_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'SHIP_COST_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Selected_Exceptions, 'Restated_SHIP_COST_AMOUNT']))

        _Sales_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'SALES_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'Restated_SALES_AMOUNT']))
        _Discount1_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'DISCOUNT1_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'Restated_DISCOUNT1_AMOUNT']))
        _Discount2_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'DISCOUNT2_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'Restated_DISCOUNT2_AMOUNT']))
        _Shipping_Inference_Restatement_Amount.append(sum(Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'SHIP_COST_AMOUNT'] - Seeded_Errors_Data.loc[Skipper_Similar_Exceptions, 'Restated_SHIP_COST_AMOUNT']))

        _Remaining_Sales_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_SALES_AMOUNT'] - Seeded_Errors_Data['Original_SALES_AMOUNT']))
        _Remaining_Discount1_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_DISCOUNT1_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT1_AMOUNT']))
        _Remaining_Discount2_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_DISCOUNT2_AMOUNT'] - Seeded_Errors_Data['Original_DISCOUNT2_AMOUNT']))
        _Remaining_Shipping_Misstatement_Amount.append(sum(Seeded_Errors_Data['Restated_SHIP_COST_AMOUNT'] - Seeded_Errors_Data['Original_SHIP_COST_AMOUNT']))
        
        print('Completed Round ' + str(Simulation) + ' out of ' + str(Rounds) + ' for Similarity_Threshold of ' + str(Similarity_Threshold) + '!!!!!',  end = "\r")
        Simulation += 1
    Similarity_Threshold = round(Similarity_Threshold + 0.002, 3)
    Simulation = 1

In [6]:
Output = pd.DataFrame()
Output['Similarity Threshold'] = _Similarity_Threshold
Output['Simulation'] = _Simulation
Output['Investigated Exceptions'] = _Investigated_Exceptions_Count
Output['Similar Exceptions'] = _Similar_Exceptions_Count
Output['Lowest Risk Selected'] = _Lowest_Risk_Selected
Output['True Positive Inference'] = _Positive_Inference_Count
Output['False Positive Inference'] = _False_Positive_Inference_Count
Output['True Negative Inference'] = _Negative_Inference_Count
Output['False Negative Inference'] = _False_Negative_Inference_Count
Output['Sales Misstatement'] = _Sales_Misstatement_Amount
Output['Discount1 Misstatement'] = _Discount1_Misstatement_Amount
Output['Discount2 Misstatement'] = _Discount2_Misstatement_Amount
Output['Shipping Misstatement'] = _Shipping_Misstatement_Amount
Output['Sales Investigation Restatement'] = _Sales_Investigation_Restatement_Amount
Output['Discount1 Investigation Restatement'] = _Discount1_Investigation_Restatement_Amount
Output['Discount2 Investigation Restatement'] = _Discount2_Investigation_Restatement_Amount
Output['Shipping Investigation Restatement'] = _Shipping_Investigation_Restatement_Amount
Output['Sales Inference Restatement'] = _Sales_Inference_Restatement_Amount
Output['Discount1 Inference Restatement'] = _Discount1_Inference_Restatement_Amount
Output['Discount2 Inference Restatement'] = _Discount2_Inference_Restatement_Amount
Output['Shipping Inference Restatement'] = _Shipping_Inference_Restatement_Amount
Output['Remaining Sales Misstatement'] = _Remaining_Sales_Misstatement_Amount
Output['Remaining Discount1 Misstatement'] = _Remaining_Discount1_Misstatement_Amount
Output['Remaining Discount2 Misstatement'] = _Remaining_Discount2_Misstatement_Amount
Output['Remaining Shipping Misstatement'] = _Remaining_Shipping_Misstatement_Amount

In [7]:
Output

,Similarity Threshold,Simulation,Investigated Exceptions,Similar Exceptions,Lowest Risk Selected,True Positive Inference,False Positive Inference,True Negative Inference,False Negative Inference,Sales Misstatement,...,Discount2 Investigation Restatement,Shipping Investigation Restatement,Sales Inference Restatement,Discount1 Inference Restatement,Discount2 Inference Restatement,Shipping Inference Restatement,Remaining Sales Misstatement,Remaining Discount1 Misstatement,Remaining Discount2 Misstatement,Remaining Shipping Misstatement
0,0.00,1,100,0,1080.875050,0,0,0,0,2450000.0,...,187500.0,1300.0,0.0,0.0,0.0,0.0,1260000.0,2.000000e+05,187500.0,28700.0
1,0.00,2,100,0,1085.548131,0,0,0,0,2450000.0,...,187500.0,7100.0,0.0,0.0,0.0,0.0,1260000.0,2.750000e+05,187500.0,22900.0
2,0.00,3,100,0,1089.284044,0,0,0,0,2450000.0,...,182500.0,5200.0,0.0,0.0,0.0,0.0,1064000.0,2.750000e+05,192500.0,24800.0
3,0.00,4,100,0,1105.470275,0,0,0,0,2450000.0,...,100000.0,9900.0,0.0,0.0,0.0,0.0,1260000.0,2.750000e+05,275000.0,20100.0
4,0.00,5,100,0,1120.986976,0,0,0,0,2450000.0,...,100000.0,13700.0,0.0,0.0,0.0,0.0,1260000.0,2.750000e+05,275000.0,16300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,0.02,96,100,152,726.987424,66,1,66,19,2450000.0,...,107500.0,2400.0,1862000.0,217500.0,217500.0,48700.0,-42000.0,5.000000e+04,50000.0,-21100.0
1096,0.02,97,100,153,716.665439,61,0,71,21,2450000.0,...,115000.0,6100.0,1932000.0,255000.0,210000.0,42800.0,28000.0,7.251977e-13,50000.0,-18900.0
1097,0.02,98,100,154,690.270450,67,0,72,15,2450000.0,...,65000.0,3700.0,2072000.0,190000.0,260000.0,41300.0,-56000.0,5.000000e+04,50000.0,-15000.0
1098,0.02,99,100,151,690.393798,64,0,71,16,2450000.0,...,105000.0,3800.0,2156000.0,210000.0,220000.0,39800.0,-112000.0,8.750000e+04,50000.0,-13600.0


In [8]:
column_names = list(Output.columns[1:42])
column_names1 = column_names[1:]

In [9]:
Mean_Summary = pd.pivot_table(Output, values=column_names, columns = ['Similarity Threshold'], aggfunc= 'mean').reindex(column_names)
Std_Summary = pd.pivot_table(Output, values=column_names, columns = ['Similarity Threshold'], aggfunc= 'std').reindex(column_names)

In [10]:
with pd.ExcelWriter('Skipper Results ' + str(datetime.datetime.now())[:10] + '.xlsx') as writer:
    Output.to_excel(writer, sheet_name='Output')
    Mean_Summary.to_excel(writer, sheet_name='Mean Summary')
    Std_Summary.to_excel(writer, sheet_name='Std Summary')